# 03 — Anomaly Detection (Unsupervised)
## Wind Turbine Gearbox Anomaly Detection

Bu notebook, **etiket kullanmadan** (unsupervised) anomali tespiti yöntemlerini uygular. Gerçek dünya senaryolarında etiketli veri çoğunlukla mevcut değildir.

**Yöntemler:**
1. Isolation Forest
2. One-Class SVM
3. Local Outlier Factor (LOF)
4. Autoencoder (Keras)

Her yöntem için anomali skoru üretilir ve gerçek etiketlerle karşılaştırılır.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import glob
import warnings
warnings.filterwarnings('ignore')

from sklearn.ensemble import IsolationForest
from sklearn.svm import OneClassSVM
from sklearn.neighbors import LocalOutlierFactor
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    roc_auc_score, average_precision_score,
    precision_recall_curve, roc_curve
)

plt.rcParams['figure.figsize'] = (12, 6)
sns.set_style('whitegrid')
print('Libraries loaded!')

# Ensure results directory exists
RESULTS_DIR = "results"
os.makedirs(RESULTS_DIR, exist_ok=True)
print(f"Results will be saved to: {os.path.abspath(RESULTS_DIR)}")


## 1. Veri Yükleme

In [ ]:
PROCESSED_PATH = '../data/processed/features_engineered.csv'
DATA_PATH = '/kaggle/input/wind-turbine-gearbox-anomaly-detection-5year-scada/'

if os.path.exists(PROCESSED_PATH):
    df = pd.read_csv(PROCESSED_PATH, index_col=0, parse_dates=True)
else:
    csv_files = glob.glob(os.path.join(DATA_PATH, '*.csv'))
    dfs = [pd.read_csv(f) for f in sorted(csv_files)]
    if not csv_files:
        raise FileNotFoundError(
            f'No CSV files found in {DATA_PATH}. '
            'Run notebook 01 first or ensure the dataset is mounted.'
        )
    df = pd.concat(dfs, ignore_index=True)
    time_col = [c for c in df.columns if 'time' in c.lower() or 'date' in c.lower()]
    if time_col:
        df[time_col[0]] = pd.to_datetime(df[time_col[0]])
        df = df.sort_values(time_col[0]).set_index(time_col[0])

anomaly_col = [c for c in df.columns if 'anomal' in c.lower() or 'label' in c.lower() or 'fault' in c.lower() or 'alarm' in c.lower() or 'fail' in c.lower() or 'error' in c.lower() or 'status' in c.lower()]
ANOMALY_COL = anomaly_col[0] if anomaly_col else df.columns[-1]
# Ensure the anomaly column is binary (0/1);
# if values are continuous/multi-class, binarize: any non-zero → 1
_unique = df[ANOMALY_COL].dropna().unique()
if not set(_unique).issubset({0, 1, 0.0, 1.0, True, False}):
    print(f'Warning: {ANOMALY_COL!r} has non-binary values {sorted(_unique)[:5]}...'
          ' — binarizing (0=normal, >0=anomaly).')
    df[ANOMALY_COL] = (df[ANOMALY_COL] != 0).astype(int)

# Feature ve target
X = df.select_dtypes(include=[np.number]).drop(columns=[ANOMALY_COL], errors='ignore')
y = df[ANOMALY_COL].astype(int)
X = X.replace([np.inf, -np.inf], np.nan).fillna(0)

# Ölçeklendirme
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Contamination (anomali oranı)
contamination = float(y.mean())
print(f'Dataset shape: {X.shape}')
print(f'Contamination rate: {contamination:.4f} ({contamination*100:.2f}%)')

## 2. Isolation Forest

Isolation Forest, anomalileri izole etmek için rastgele ağaçlar kullanır. Anomaliler daha az sayıda bölümleme ile izole edildiğinden daha kısa yollara sahip olur.

**Contamination tuning:** Gerçek anomali oranını contamination parametresi olarak kullanıyoruz.

In [ ]:
# Contamination tuning
contamination_values = [0.01, 0.02, 0.05, 0.1, contamination]
if_results = []

for cont in contamination_values:
    if_model = IsolationForest(n_estimators=200, contamination=cont,
                               random_state=42, n_jobs=-1)
    if_labels = -if_model.fit_predict(X_scaled).astype(float)  # -1 anomaly → 1
    # Normalize to [0, 1] score
    if_anomaly_score = -if_model.score_samples(X_scaled)
    if_anomaly_score = (if_anomaly_score - if_anomaly_score.min()) / \
                       (if_anomaly_score.max() - if_anomaly_score.min())
    
    auc = roc_auc_score(y, if_anomaly_score)
    ap = average_precision_score(y, if_anomaly_score)
    if_results.append({'contamination': cont, 'ROC-AUC': auc, 'PR-AUC': ap})

if_df = pd.DataFrame(if_results)
print('Isolation Forest — Contamination Tuning:')
print(if_df.round(4))

# Best contamination
best_cont = if_df.loc[if_df['ROC-AUC'].idxmax(), 'contamination']
print(f'\nBest contamination: {best_cont}')

# Final model
if_final = IsolationForest(n_estimators=200, contamination=best_cont,
                           random_state=42, n_jobs=-1)
if_final.fit(X_scaled)
if_scores_final = -if_final.score_samples(X_scaled)
if_scores_final = (if_scores_final - if_scores_final.min()) / \
                  (if_scores_final.max() - if_scores_final.min())

print(f'Final IF — ROC-AUC: {roc_auc_score(y, if_scores_final):.4f}')

## 3. One-Class SVM

One-Class SVM, sadece normal veriden öğrenir ve yeni veriyi bu sınıra göre değerlendirir. Kernel trick ile yüksek boyutlu uzayda anomali sınırı çizer.

In [ ]:
# One-Class SVM — büyük veri setleri için örneklem
SAMPLE_SIZE = min(10000, len(X_scaled))
sample_idx = np.random.choice(len(X_scaled), SAMPLE_SIZE, replace=False)
sample_idx.sort()

X_sample = X_scaled[sample_idx]
y_sample = y.iloc[sample_idx]

# Normal örnekler üzerinde eğit
normal_mask = y_sample == 0
X_normal = X_sample[normal_mask]

print(f'Training One-Class SVM on {len(X_normal):,} normal samples...')
oc_svm = OneClassSVM(kernel='rbf', gamma='scale', nu=contamination)
oc_svm.fit(X_normal)

# Tüm örnekler için skor
svm_scores = -oc_svm.score_samples(X_sample)
svm_scores = (svm_scores - svm_scores.min()) / (svm_scores.max() - svm_scores.min())

svm_auc = roc_auc_score(y_sample, svm_scores)
svm_ap = average_precision_score(y_sample, svm_scores)
print(f'One-Class SVM — ROC-AUC: {svm_auc:.4f} | PR-AUC: {svm_ap:.4f}')

## 4. Local Outlier Factor (LOF)

LOF, veri noktasının komşularına göre yerel yoğunluğunu ölçer. Düşük yerel yoğunluk = potansiyel anomali.

In [ ]:
print(f'Computing LOF on {SAMPLE_SIZE:,} samples...')
lof = LocalOutlierFactor(
    n_neighbors=20,
    contamination=contamination,
    novelty=True,
    n_jobs=-1
)
lof.fit(X_normal)

lof_scores = -lof.score_samples(X_sample)
lof_scores = (lof_scores - lof_scores.min()) / (lof_scores.max() - lof_scores.min())

lof_auc = roc_auc_score(y_sample, lof_scores)
lof_ap = average_precision_score(y_sample, lof_scores)
print(f'LOF — ROC-AUC: {lof_auc:.4f} | PR-AUC: {lof_ap:.4f}')

## 5. Autoencoder (Keras)

Autoencoder, veriyi düşük boyutlu bir temsile sıkıştırıp geri açar. Normal veri için düşük reconstruction error, anomali için yüksek reconstruction error beklenir.

In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

tf.random.set_seed(42)
print(f'TensorFlow version: {tf.__version__}')

# Autoencoder mimarisi
input_dim = X_scaled.shape[1]
encoding_dim = max(8, input_dim // 4)

def build_autoencoder(input_dim, encoding_dim):
    """Derin autoencoder mimarisi."""
    inputs = keras.Input(shape=(input_dim,))
    
    # Encoder
    x = layers.Dense(input_dim // 2, activation='relu')(inputs)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.2)(x)
    x = layers.Dense(encoding_dim * 2, activation='relu')(x)
    encoded = layers.Dense(encoding_dim, activation='relu', name='bottleneck')(x)
    
    # Decoder
    x = layers.Dense(encoding_dim * 2, activation='relu')(encoded)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.2)(x)
    x = layers.Dense(input_dim // 2, activation='relu')(x)
    decoded = layers.Dense(input_dim, activation='linear', name='reconstruction')(x)
    
    autoencoder = keras.Model(inputs, decoded)
    encoder = keras.Model(inputs, encoded)
    return autoencoder, encoder

autoencoder, encoder = build_autoencoder(input_dim, encoding_dim)
autoencoder.compile(optimizer='adam', loss='mse')
autoencoder.summary()

In [ ]:
# Sadece normal veriler üzerinde eğit
X_normal_all = X_scaled[y.values == 0]
split = int(len(X_normal_all) * 0.8)
X_ae_train = X_normal_all[:split]
X_ae_val = X_normal_all[split:]

print(f'Autoencoder training data: {X_ae_train.shape}')

callbacks = [
    keras.callbacks.EarlyStopping(monitor='val_loss', patience=10,
                                   restore_best_weights=True),
    keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5,
                                       patience=5, min_lr=1e-6)
]

history = autoencoder.fit(
    X_ae_train, X_ae_train,
    validation_data=(X_ae_val, X_ae_val),
    epochs=50, batch_size=256,
    callbacks=callbacks, verbose=1
)

In [ ]:
# Training history
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(history.history['loss'], label='Train Loss', linewidth=2)
ax.plot(history.history['val_loss'], label='Val Loss', linewidth=2)
ax.set_title('Autoencoder Training History', fontsize=13)
ax.set_xlabel('Epoch')
ax.set_ylabel('MSE Loss')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('results/autoencoder_training.png', dpi=150, bbox_inches='tight')
plt.show()

# Reconstruction error skoru
X_reconstructed = autoencoder.predict(X_scaled, batch_size=512, verbose=0)
ae_scores = np.mean(np.square(X_scaled - X_reconstructed), axis=1)
ae_scores_norm = (ae_scores - ae_scores.min()) / (ae_scores.max() - ae_scores.min())

ae_auc = roc_auc_score(y, ae_scores_norm)
ae_ap = average_precision_score(y, ae_scores_norm)
print(f'Autoencoder — ROC-AUC: {ae_auc:.4f} | PR-AUC: {ae_ap:.4f}')

## 6. Anomali Skoru Dağılımı Görselleştirmesi

In [ ]:
score_dict = {
    'Isolation Forest': (if_scores_final, y.values),
    'One-Class SVM': (svm_scores, y_sample.values),
    'LOF': (lof_scores, y_sample.values),
    'Autoencoder': (ae_scores_norm, y.values)
}

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

for i, (name, (scores, labels)) in enumerate(score_dict.items()):
    ax = axes[i]
    scores_normal = scores[labels == 0]
    scores_anomaly = scores[labels == 1]
    
    ax.hist(scores_normal, bins=50, alpha=0.6, color='#2ecc71', density=True, label='Normal')
    ax.hist(scores_anomaly, bins=50, alpha=0.6, color='#e74c3c', density=True, label='Anomaly')
    
    ax.set_title(f'{name} — Anomaly Score Distribution', fontsize=12)
    ax.set_xlabel('Anomaly Score')
    ax.set_ylabel('Density')
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('results/anomaly_score_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. ROC ve PR Karşılaştırması

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
colors = ['#e74c3c', '#3498db', '#2ecc71', '#9b59b6']

for (name, (scores, labels)), color in zip(score_dict.items(), colors):
    fpr, tpr, _ = roc_curve(labels, scores)
    auc = roc_auc_score(labels, scores)
    axes[0].plot(fpr, tpr, color=color, label=f'{name} ({auc:.3f})', linewidth=2)
    
    precision, recall, _ = precision_recall_curve(labels, scores)
    ap = average_precision_score(labels, scores)
    axes[1].plot(recall, precision, color=color, label=f'{name} ({ap:.3f})', linewidth=2)

axes[0].plot([0,1], [0,1], 'k--', linewidth=1)
axes[0].set_title('ROC Curves — Unsupervised Methods', fontsize=13)
axes[0].set_xlabel('FPR'); axes[0].set_ylabel('TPR')
axes[0].legend(fontsize=9); axes[0].grid(True, alpha=0.3)

axes[1].set_title('PR Curves — Unsupervised Methods', fontsize=13)
axes[1].set_xlabel('Recall'); axes[1].set_ylabel('Precision')
axes[1].legend(fontsize=9); axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('results/unsupervised_roc_pr.png', dpi=150, bbox_inches='tight')
plt.show()

## 8. Hangi Sensörler En Çok Katkı Sağlıyor?

In [ ]:
# Anomali skorları ile özellik korelasyonu
feature_names = X.columns.tolist()
if_scores_df = pd.Series(if_scores_final, index=df.index, name='IF_Score')

# Her özellik ile anomali skoru arasındaki korelasyon
original_features = [c for c in feature_names if '_roll_' not in c and '_lag_' not in c 
                     and '_sin_' not in c and '_cos_' not in c]

if original_features:
    corr_with_score = df[original_features].corrwith(if_scores_df).abs().sort_values(ascending=False)
    
    fig, ax = plt.subplots(figsize=(10, 6))
    top_n = min(15, len(corr_with_score))
    colors = ['#e74c3c' if v > 0.3 else '#3498db' for v in corr_with_score.head(top_n)]
    corr_with_score.head(top_n).plot(kind='barh', ax=ax, color=colors[::-1])
    ax.set_title('Sensor Correlation with Anomaly Score (Isolation Forest)', fontsize=13)
    ax.set_xlabel('|Correlation|')
    ax.axvline(x=0.3, color='red', linestyle='--', label='Threshold=0.3')
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig('results/sensor_anomaly_contribution.png', dpi=150, bbox_inches='tight')
    plt.show()

## 9. Özet ve Sonuçlar

In [ ]:
# Sonuç tablosu
unsupervised_results = [
    {'Method': 'Isolation Forest', 'ROC-AUC': roc_auc_score(y, if_scores_final),
     'PR-AUC': average_precision_score(y, if_scores_final)},
    {'Method': 'One-Class SVM', 'ROC-AUC': svm_auc, 'PR-AUC': svm_ap},
    {'Method': 'LOF', 'ROC-AUC': lof_auc, 'PR-AUC': lof_ap},
    {'Method': 'Autoencoder', 'ROC-AUC': ae_auc, 'PR-AUC': ae_ap}
]

results_df = pd.DataFrame(unsupervised_results).set_index('Method')
print('=== UNSUPERVISED METHOD COMPARISON ===')
print(results_df.round(4))

# Görselleştir
fig, ax = plt.subplots(figsize=(10, 4))
results_df.plot(kind='bar', ax=ax, color=['#3498db', '#e74c3c'], alpha=0.8)
ax.set_title('Unsupervised Methods — ROC-AUC and PR-AUC Comparison', fontsize=13)
ax.set_ylabel('Score')
ax.set_ylim(0, 1)
ax.tick_params(axis='x', rotation=30)
ax.legend()
ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.savefig('results/unsupervised_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

# Anomali skorlarını kaydet (Hybrid ensemble için)
os.makedirs('../data/processed', exist_ok=True)
anomaly_scores_df = pd.DataFrame({
    'if_score': if_scores_final,
    'ae_score': ae_scores_norm
}, index=df.index)
anomaly_scores_df.to_csv('../data/processed/unsupervised_scores.csv')
print('Anomaly scores saved!')
print('\n✅ Unsupervised Anomaly Detection Complete!')
# Karşılaştırma tablosunu kaydet
results_df.to_csv(f"{RESULTS_DIR}/unsupervised_comparison.csv")
print("Saved: unsupervised_comparison.csv")

# Anomali skorlarını da kaydet (tüm yöntemler)
all_scores_df = pd.DataFrame({
    "if_score": if_scores_final,
    "ae_score": ae_scores_norm,
    "y_true": y.values
}, index=df.index)
all_scores_df.to_csv(f"{RESULTS_DIR}/all_anomaly_scores.csv")
print("Saved: all_anomaly_scores.csv")


## Özet

| Yöntem | Avantaj | Dezavantaj |
|--------|---------|------------|
| Isolation Forest | Hızlı, ölçeklenebilir | Küresel anomali varsayımı |
| One-Class SVM | Güçlü sınır | Büyük veride yavaş |
| LOF | Yerel yoğunluk | Yüksek boyutta zayıf |
| Autoencoder | Kompleks pattern | Eğitim süresi uzun |

**Sonraki Adım:** `04_TimeSeries_DeepLearning` — zaman bağımlılığını modelleyen derin öğrenme